将**工业级分类流水线的严谨架构**与**GAN对抗训练的复杂逻辑**结合起来，正是深度学习工程化落地的重要一步。设计了一个“工业级标准配置的 GAN 训练框架”。

### 一、 示例总括说明

本示例实现了一个**基于 PyTorch Lightning 的企业级 GAN 训练与推理流水线**。
它不仅包含了生成器与判别器的零和博弈（Minimax Game）逻辑，还完全继承了分类流水线中的工程化规范。代码彻底解耦了数据处理与模型计算，引入了判别器准确率监控（TorchMetrics）、多优化器与多学习率调度系统、丰富的 Callback 体系（自动保存权重、生命周期注入、图像生成可视化），并配置了混合精度与梯度累加来榨干有限的硬件资源，最终展示了如何利用训练好的生成器进行纯前向推理。

### 二、 涵盖的综合核心知识点

本示例毫无保留地融合了你提到的所有核心技术点：

1. **LightningDataModule 数据解耦**：封装 `prepare_data` 和 `setup`，分离训练集与预测集（用于纯生成阶段）。
2. **完全生命周期覆盖 (LightningModule)**：重写了 `training_step` 用于对抗训练，以及 `predict_step` 用于脱离训练上下文后的批量图像生成。
3. **手动控制与多优化器 (Manual Optimization)**：通过 `self.automatic_optimization = False` 接管控制权，在 `training_step` 中获取 `g_opt` 和 `d_opt`，交替执行前向传播、清空梯度、`manual_backward()` 和权重更新。
4. **分布式指标监控 (TorchMetrics)**：引入 `BinaryAccuracy` 计算判别器在真实图像和假图像上的识别准确率，开启 `sync_dist=True` 防止多卡指标漂移，借此监控 GAN 是否发生模式崩溃（Mode Collapse）。
5. **复杂日志体系与自适应监控**：使用 `log_dict` 批量记录 $G$ 损失、$D$ 损失以及 $D$ 的准确率到 TensorBoard。
6. **联合学习率调度系统 (`configure_optimizers`)**：同时为生成器和判别器配置 Adam 优化器，并分别绑定 `CosineAnnealingLR` 学习率调度器，展示高级配置语法。
7. **综合 Callback 矩阵**：
* **内置**：使用 `ModelCheckpoint` 定期并自动保存最新模型。
* **自定义1**：`LifecycleLoggerCallback` 监控运行阶段。
* **自定义2**：`ImageSamplerCallback` 随 Epoch 动态保存生成图像。


8. **硬件极致优化 (Trainer 设定)**：配置 `16-mixed` 混合精度与 `accumulate_grad_batches`。代码中的参数（如适中的 Batch Size 与梯度累加）特别契合 4GB 显存设备（如 GTX 1050 Ti 等环境）的本地开发验证。


In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision import transforms
import torchvision

# 导入 PyTorch Lightning 核心组件与回调机制
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, Callback
# 导入 W&B (Weights & Biases) 日志记录器
from lightning.pytorch.loggers import WandbLogger
import wandb 

# 导入 TorchMetrics 用于高效且支持分布式同步的指标计算
from torchmetrics.classification import BinaryAccuracy

# ==========================================
# 1. 自定义 Callbacks：生命周期记录 & 图像可视化
# ==========================================
class LifecycleLoggerCallback(Callback):
    """
    自定义生命周期回调函数：
    利用 Lightning 的 Hook (钩子) 机制，在训练启动时输出系统状态，
    实现业务逻辑与核心模型代码的彻底解耦。
    """
    def on_fit_start(self, trainer, pl_module):
        # 训练开始时触发，打印当前硬件加速器类型
        device_str = str(trainer.strategy.root_device).upper()
        print(f"\n[🚀 架构就绪] 工业级 GAN 流水线启动！设备: {device_str}")
        print(f"[📊 日志系统] Weights & Biases (W&B) 记录已开启，请前往浏览器查看大屏面板。")

class ImageSamplerCallback(Callback):
    """
    自动图像采样回调函数：
    在训练过程中定期（每个 Epoch 结束时）使用固定的隐向量（噪声）生成图像，
    并直接将其推送到 W&B 云端进行可视化，用于直观监控 GAN 的演进过程。
    """
    def __init__(self, num_samples: int = 16, latent_dim: int = 100):
        super().__init__()
        self.num_samples = num_samples   # 采样图像的数量（默认生成 16 张组合成 4x4 网格）
        self.latent_dim = latent_dim     # 输入噪声向量的维度
        self.fixed_noise = None          # 固定噪声，保证每轮评估时输入相同，以便对比生成质量的演进

    def on_fit_start(self, trainer, pl_module):
        # 在整个 Fit 流程开始时，在模型所在的设备上初始化一组固定的高斯噪声
        self.fixed_noise = torch.randn(self.num_samples, self.latent_dim, device=pl_module.device)

    def on_train_epoch_end(self, trainer, pl_module):
        # 进入评估状态：切换模型至 eval 模式，关闭 BatchNorm/Dropout 的动态更新
        pl_module.eval()
        with torch.no_grad():
            # 使用固定噪声通过生成器进行前向推理（显式调用前向传播）
            fake_images = pl_module(self.fixed_noise)
            # 将生成的批量单通道图片 [B, 1, 28, 28] 拼接成一张 4x4 的大型网格图，并归一化至 [0, 1]
            grid = torchvision.utils.make_grid(fake_images, nrow=4, normalize=True)
            
            # 提取 trainer 中配置的 W&B 实验运行实例 (wandb.run)
            # 使用 wandb.Image 包装多维 Tensor，并附带当前 Epoch 作为标签推送到云端仪表盘
            trainer.logger.experiment.log({
                "Generated_Evolution": [wandb.Image(grid, caption=f"Epoch {trainer.current_epoch}")]
            })
        # 恢复训练状态：恢复模型为 train 模式
        pl_module.train()

# ==========================================
# 2. 数据解耦：LightningDataModule
# ==========================================
class MNISTDataModule(pl.LightningDataModule):
    """
    标准化数据模块：
    将数据下载、预处理、训练集划分以及 DataLoader 的生命周期管理高度封装，
    保证了同一套数据模块可以无缝复用到不同的模型架构中。
    """
    def __init__(self, data_dir: str = "./data", batch_size: int = 64):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        # GAN 图像数据的特殊预处理：
        # 常用策略是将图像像素值归一化到 [-1, 1] 区间（均值 0.5，方差 0.5），
        # 从而完美匹配生成器最后一层 Tanh 激活函数的输出范围。
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def prepare_data(self):
        # 专门用于处理 CPU/单进程 任务，如文件下载或磁盘写入
        # 分布式训练（多卡）时，该方法只会在主卡（Rank 0）上执行一次，避免重复写盘崩溃
        MNIST(self.data_dir, train=True, download=True)

    def setup(self, stage: str = None):
        # 专门用于处理 GPU/多进程 任务，在每张显卡上独立执行，构建内部 Dataset 状态
        if stage == "fit" or stage is None:
            self.train_ds = MNIST(self.data_dir, train=True, transform=self.transform)
        if stage == "predict":
            # 模拟生产环境：纯推理/生成阶段不需要真实的 MNIST 数据集
            # 这里构建一个 100 行的虚拟占位数据，仅用来控制 trainer.predict 循环的迭代批次
            self.predict_ds = torch.zeros(100, 1) 

    def train_dataloader(self):
        # 返回标准的训练加载器。drop_last=True 非常关键，能避免最后一个非完整 Batch 引发尺寸不匹配问题
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=4, drop_last=True)
        
    def predict_dataloader(self):
        # 返回推理阶段的数据加载器
        return DataLoader(self.predict_ds, batch_size=16, shuffle=False)

# ==========================================
# 3. 核心模型：LitIndustrialGAN
# ==========================================
class LitIndustrialGAN(pl.LightningModule):
    """
    企业级生成对抗网络核心类：
    打破传统的“重叠 For 循环”写法，将对抗博弈逻辑完全集成在单个类中。
    """
    def __init__(self, latent_dim: int = 100, lr: float = 0.0002):
        super().__init__()
        # 自动将构造函数的参数（latent_dim, lr）保存到 self.hparams 中，便于在网络各处调用
        self.save_hyperparameters()
        
        # 【关键技术点】：必须关闭 Lightning 的自动优化机制 (BPTT/Standard Flow)
        # 因为 GAN 包含多优化器，需要交替反向传播（更新 G 的权值时固定 D，更新 D 的权值时固定 G），
        # 必须在 training_step 中通过手工控制梯度流。
        self.automatic_optimization = False

        # --- 生成器网络 (Generator) ---
        # 作用：将一个 [B, latent_dim] 的噪声向量逐步上采样、映射并放大为 [B, 784] 的连续分布
        self.generator = nn.Sequential(
            nn.Linear(self.hparams.latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 28 * 28),
            nn.Tanh() # 输出层采用 Tanh 激活，将像素数值牢牢锁定在 [-1, 1] 之间
        )

        # --- 判别器网络 (Discriminator) ---
        # 作用：接收一张扁平化后的图像 [B, 784]，输出其为“真图”的概率得分 [B, 1]
        self.discriminator = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            # nn.Sigmoid() # 输出层采用 Sigmoid 激活，将结果映射为 0~1 的置信概率
        )

        # 【关键技术点】：引入 TorchMetrics 的分类准确率指标
        # 分别建立两个实例监控判别器对真实样本和假样本的识别率。
        # 这样做能极快诊断出 GAN 训练中的两类重大危机：判别器过强导致生成器无法学习，或发生模式崩溃 (Mode Collapse)。
        self.d_acc_real = BinaryAccuracy()
        self.d_acc_fake = BinaryAccuracy()

        # 用于ONNX 导出时的输入示例张量，他会自己动被注册为模型属性，并在导出时自动使用，确保 ONNX 模型的输入输出接口完全正确。
        self.example_input_array = torch.randn(1, 100)

    def forward(self, z):
        """定义标准前向传播接口：通常指定为生成器的纯推理路径"""
        img = self.generator(z)
        # 将展平的 [B, 784] 自动变形回二维的规范图像张量 [B, 1, 28, 28]
        return img.view(img.size(0), 1, 28, 28)

    def adversarial_loss(self, y_hat, y):
        """定义对抗的核心损失：经典的二元交叉熵损失函数 (BCE Loss),带 logits 的安全版本"""
        return F.binary_cross_entropy_with_logits(y_hat, y)

    def training_step(self, batch, batch_idx):
        """
        手动优化流下的单步对抗迭代逻辑。
        """
        real_imgs, _ = batch # GAN 是无监督/自监督生成，不需要 MNIST 数据集的分类数字标签
        batch_size = real_imgs.size(0)
        # 将真实图像 [B, 1, 28, 28] 展平为 [B, 784] 以输入到判别器的全连接层
        real_imgs_flat = real_imgs.view(batch_size, -1)

        # 【关键技术点】：通过 self.optimizers() 手工提取在 configure_optimizers 中声明的优化器
        opt_g, opt_d = self.optimizers()
        
        # 预先构建目标真假标签：真实图对应 1.0，生成图对应 0.0
        valid_labels = torch.ones(batch_size, 1, device=self.device)
        fake_labels = torch.zeros(batch_size, 1, device=self.device)
        
        # 从标准正态分布中抽取本轮迭代的潜在空间隐向量 z
        z = torch.randn(batch_size, self.hparams.latent_dim, device=self.device)

        # ----------------------------------------------------------------------
        # 阶段 1: 训练生成器 (Generator) 
        # 目标：让生成器输出的伪造图像在被判别器检验时，得到的判别得分越接近 1 (真图) 越好
        # ----------------------------------------------------------------------
        # 1. 产生伪造图像
        fake_imgs = self.generator(z)
        # 2. 将未阻断梯度的伪造图像送入判别器，计算对抗损失
        g_loss = self.adversarial_loss(self.discriminator(fake_imgs), valid_labels)

        # 3. 手工管理反向传播三部曲（更新 Generator）
        opt_g.zero_grad()               # 清空生成器历史梯度
        self.manual_backward(g_loss)    # 执行反向求导计算
        opt_g.step()                    # 更新生成器参数权重

        # ----------------------------------------------------------------------
        # 阶段 2: 训练判别器 (Discriminator)
        # 目标：精确区分真实图像（判别为 1）与刚刚生成的伪造图像（判别为 0）
        # ----------------------------------------------------------------------
        # 1. 计算判别器在真实图像上的损失与准确率
        d_out_real = self.discriminator(real_imgs_flat)
        d_real_loss = self.adversarial_loss(d_out_real, valid_labels)
        acc_real = self.d_acc_real(d_out_real, valid_labels)

        # 2. 计算判别器在假图像上的损失与准确率
        # 【关键技术点】：必须使用 fake_imgs.detach() 截断计算图！
        # 这样可以将假图像转换为一个纯粹的常量张量，梯度流将无法向后渗透进生成器网络，
        # 从而极大地节省显存开销，并确保此时只优化判别器的参数。
        d_out_fake = self.discriminator(fake_imgs.detach())
        d_fake_loss = self.adversarial_loss(d_out_fake, fake_labels)
        acc_fake = self.d_acc_fake(d_out_fake, fake_labels)

        # 3. 判别器的联合平均损失
        d_loss = (d_real_loss + d_fake_loss) / 2

        # 4. 手工管理反向传播三部曲（更新 Discriminator）
        opt_d.zero_grad()               # 清空判别器历史梯度
        self.manual_backward(d_loss)    # 执行反向求导计算
        opt_d.step()                    # 更新判别器参数权重

        # ----------------------------------------------------------------------
        # 阶段 3: 批量指标上报与日志记录
        # ----------------------------------------------------------------------
        # 【关键技术点】：利用 log_dict 聚合日志，并在分布式（多卡 DDP）环境中开启 sync_dist=True，
        # 这样 Lightning 会在 Epoch 结束时自动跨卡聚合所有进程的 Loss 和 Accuracy，保证日志绝对不失真。
        metrics_dict = {
            "train/g_loss": g_loss,
            "train/d_loss": d_loss,
            "train/d_acc_real": acc_real,
            "train/d_acc_fake": acc_fake
        }
        self.log_dict(metrics_dict, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        """
        【工业部署接口】：模拟生产环境下的离线/在线纯前向生成推理。
        当调用 trainer.predict() 时触发，完全脱离反向传播上下文，只负责根据输入请求批量吐出图片。
        """
        num_images = batch.size(0) 
        # 为当前批次的生成请求实时构造随机噪声
        z = torch.randn(num_images, self.hparams.latent_dim, device=self.device)
        # 调用模型的 forward 前向传播，得到 [B, 1, 28, 28] 的生成图
        generated_imgs = self(z)
        return generated_imgs

    def configure_optimizers(self):
        """
        多优化器与多学习率调度链条配置：
        """
        lr = self.hparams.lr
        # 为生成器与判别器分别定义 Adam 优化器。GAN 中经典超参设置：betas=(0.5, 0.999) 能有效平抑对抗震荡
        opt_g = torch.optim.Adam(self.generator.parameters(), lr=lr, betas=(0.5, 0.999))
        opt_d = torch.optim.Adam(self.discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
        
        # 引入工业标准的余弦退火学习率调度器 (CosineAnnealingLR)，模拟学习率平滑衰减并防止过早陷入死局
        sch_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=10)
        sch_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=10)
        
        # 将优化器与对应的调度器打包成标准字典格式返回。
        # interval: 'epoch' 意味着每个训练 epoch 结束时自动执行一次 scheduler.step()
        return (
            {"optimizer": opt_g, "lr_scheduler": {"scheduler": sch_g, "interval": "epoch"}},
            {"optimizer": opt_d, "lr_scheduler": {"scheduler": sch_d, "interval": "epoch"}}
        )

/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/lightning/fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [3]:

# ==========================================
# 4. 组装与运行中枢
# ==========================================

# 固定全局随机种子，确保权重初始化、噪声分布以及数据打乱结果在多次运行时完全可复现
pl.seed_everything(42)

# 实例化数据模块与生成对抗网络模型
datamodule = MNISTDataModule(batch_size=64) 
model = LitIndustrialGAN(latent_dim=100)

# 【W&B 核心配置】：指定云端专案组名称 (project) 以及当前独立实验的标称 (name)
logger = WandbLogger(entity="Jerry-Auto",project="learn_wandb", name="MNIST_Baseline")


# 配置内置的自动权重检查点保存策略
# 由于 GAN 的演进极其脆弱，没有传统意义上的 val_loss 供其早停或筛选，
# 工业界最通用的做法是设定 every_n_epochs 周期性保存最新进度，防止设备断电或崩溃。
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints_gan",
    filename="gan-latest-{epoch:02d}",
    save_last=True,                 # 始终覆盖并保持一份最新的 'last.ckpt' 权重
    every_n_epochs=2                # 每隔 2 个 Epoch 存档一次
)

# 构建强大的超级训练驱动器 Trainer
trainer = pl.Trainer(
    max_epochs=10,                 # 迭代上限 10 个 Epoch
    accelerator="auto",            # 自动探测可用硬件（支持 CPU、GPU、MPS 等）
    devices=1,                     # 约束单卡训练
    precision="16-mixed",          # 【性能倍增器】：开启 16 位混合精度，大幅削减显存开销（极适合 4GB VRAM 显卡）
    # accumulate_grad_batches=2,     # 【梯度累加技术】：在手动优化下可充当乘法器，等效于将 batch_size 扩展为 64 * 2 = 128
    logger=logger,                 # 绑定 W&B 云端看板
    callbacks=[
        checkpoint_callback,       # 挂载自动存档回调
        LifecycleLoggerCallback(), # 挂载生命周期打印回调
        ImageSamplerCallback()     # 挂载 W&B 图像动态生成采样回调
    ]
)

Global seed set to 42
wandb: WARNING `wandb.require('service')` is a no-op as it is now the default behavior.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zhangjinrui/.netrc.
wandb: Currently logged in as: 1763287396 (Jerry-Auto) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [11]:
print("\n========== 阶段 1：对抗训练 ==========")
# 启动全自动训练流：内部会自动串联 prepare_data, setup 并交替调用 training_step
trainer.fit(model, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type           | Params
-------------------------------------------------
0 | generator     | Sequential     | 1.5 M 
1 | discriminator | Sequential     | 533 K 
2 | d_acc_real    | BinaryAccuracy | 0     
3 | d_acc_fake    | BinaryAccuracy | 0     
-------------------------------------------------
2.0 M     Trainable params
0         Non-trainable params
2.0 M     Total params
8.092     Total estimated model params size (MB)



========== 阶段 1：对抗训练 ==========

[🚀 架构就绪] 工业级 GAN 流水线启动！设备: CUDA:0
[📊 日志系统] Weights & Biases (W&B) 记录已开启，请前往浏览器查看大屏面板。


Training: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [12]:
# 养成企业开发的优良习惯：Fit 流程结束后主动关闭当前的 W&B 云端运行实例，清空管道
wandb.finish() 

epoch,▁▂▃▃▄▅▆▆▇█
train/d_acc_fake,▇▇██▆▅▃▃▁▁
train/d_acc_real,█▅▄▃▂▂▂▂▁▁
train/d_loss,▁▃▄▅▆▇▇▇██
train/g_loss,▃▆█▇▅▃▂▃▂▁
trainer/global_step,▁▂▂▂▂▃▃▄▄▄▄▅▅▆▆▇▇▇▇█
epoch,9
train/d_acc_fake,0.67748
train/d_acc_real,0.65592
train/d_loss,0.60877
train/g_loss,0.92251


In [4]:
print("\n========== 阶段 2：生产环境推理 ==========")

# 1. 实例化一个专门用于纯推理的干净 Trainer（去掉 logger，保持与训练一致的设备和精度）
predict_trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed"  # 保持混合精度以加速推理并对齐推理环境
)

# 2. 传入这个纯净的 trainer 中执行预测
# 注意：ckpt_path="last" 需要确保你之前的 trainer 触发了 checkpoint 自动保存
predictions = predict_trainer.predict(model, datamodule=datamodule, ckpt_path="last")

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs



========== 阶段 2：生产环境推理 ==========


/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:190: UserWarning: .predict(ckpt_path="last") is set, but there is no last checkpoint available. No checkpoint will be loaded.
  rank_zero_warn(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/zhangjinrui/anaconda3/envs/flow_planner/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:442: PossibleUserWarning: The dataloader, predict_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Predicting: 0it [00:00, ?it/s]

In [5]:
# 统计从全推理流程中批量吐出的所有生成图像的总张数
total_generated = sum([batch.size(0) for batch in predictions])
print(f"\n[📦 部署完毕] 成功使用训练好的生成器，生成了 {total_generated} 张新图像。")


[📦 部署完毕] 成功使用训练好的生成器，生成了 100 张新图像。


In [ ]:
# 连 dummy_input 都不用传，Lightning 会自动用 self.example_input_array 探路
model.to_onnx(
    file_path="industrial_gan.onnx", 
    opset_version=14,
    input_names=['noise_z'],        
    output_names=['fake_images'],
    dynamic_axes={                  # 🔥 核心：告诉 ONNX 运行时，第 0 维是动态的
        'noise_z': {0: 'batch_size'},
        'fake_images': {0: 'batch_size'}
    }
)
print("✅ 完美版动态 Batch ONNX 模型导出成功！")

============= Diagnostic Run torch.onnx.export version 2.0.1+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

✅ 完美版动态 Batch ONNX 模型导出成功！


In [11]:
import torch
import onnxruntime as ort
import numpy as np

print("========== ⚙️ 开始 ONNX 推理验证 ==========")

# 1. 产生一份完全相同的随机噪声输入
batch_size = 1
noise_dim = 100
# 准备 PyTorch 的 Tensor 输入
dummy_tensor = torch.randn(batch_size, noise_dim, device=model.device)
# 准备 ONNX 的 NumPy 输入 (完全共享相同的数据)
dummy_numpy = dummy_tensor.cpu().numpy().astype(np.float32)

# 2. 获取 PyTorch 的预测结果
model.eval()
with torch.no_grad():
    pytorch_output = model(dummy_tensor).cpu().numpy()

# 3. 创建 ONNX Runtime 推理会话 (自动侦测 CUDA 或 CPU)
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession("industrial_gan.onnx", providers=providers)

# 4. 执行 ONNX 推理
# ['fake_images'] 是导出时指定的输出节点名
# {'noise_z': dummy_numpy} 是输入节点名与数据的映射
onnx_outputs = session.run(['fake_images'], {'noise_z': dummy_numpy})
onnx_output = onnx_outputs[0]

# 5. 精度对齐与验证
print(f"\n📊 结果看板:")
print(f"-> PyTorch 输出形状: {pytorch_output.shape}")
print(f"-> ONNX    输出形状: {onnx_output.shape}")

# 计算两者的最大绝对误差
max_diff = np.max(np.abs(pytorch_output - onnx_output))
print(f"-> 两者最大绝对误差 (Max Diff): {max_diff:.2e}")

# 工业级鲁棒性阈值检查 (通常低于 1e-5 视为完全一致)
if max_diff < 1e-4:
    print("\n🎉 【验证通过】ONNX 与 PyTorch 结果完美对齐！模型可安全用于流水线部署。")
else:
    print("\n⚠️ 【警告】两者存在微小点位差异，可能由于 FP16/FP32 混合精度转换导致。")

========== ⚙️ 开始 ONNX 推理验证 ==========

📊 结果看板:
-> PyTorch 输出形状: (1, 1, 28, 28)
-> ONNX    输出形状: (1, 1, 28, 28)
-> 两者最大绝对误差 (Max Diff): 1.02e-07

🎉 【验证通过】ONNX 与 PyTorch 结果完美对齐！模型可安全用于流水线部署。
